# Ejercicio 2 — Hay un bug en produccion
---

Un compañero escribió esta función para el merge incremental de orders_silver. Funciona la mayoría del tiempo, pero de vez en cuando aparecen duplicados en algunos países y el equipo de infraestructura se ha quejado de que el job tarda cada vez más.

```
import pyspark.sql.functions as F 
from delta.tables import DeltaTable

def merge_orders(spark, new_batch_df, target_table_path): 
    """ Hace merge incremental de nuevas órdenes a la tabla Silver. 
        new_batch_df: order_id_txt, customer_id, pais_cd, monto_dec, updated_ts 
    """ 
    
    target = DeltaTable.forPath(spark, target_table_path)

    # Enriquecemos con datos del cliente
    customers_df = spark.table("customers_silver") 
    enriched_df = new_batch_df.join( 
        customers_df, 
        new_batch_df.customer_id == customers_df.customer_id, 
        "left" 
    ) 

    # Bug intencional #1 
    merge_keys = "order_id_txt" 

    (target.alias("t") 
        .merge( 
            enriched_df.alias("s"), 
            f"t.{merge_keys} = s.{merge_keys}" 
        ) 
        .whenMatchedUpdateAll() 
        .whenNotMatchedInsertAll()  
        .execute() # --> ejecuta el merge
    ) 

    # Bug intencional #2 (particionamiento) 
    enriched_df.write.format("delta") \ 
        .mode("append") \  
        .partitionBy("updated_dt") \ 
        .save(target_table_path + "_backup") 

    return enriched_df
```

Dos datos que te van a servir: customers_silver todavía tiene clientes duplicados por customer_id en algunos países (viene de un histórico que no se ha limpiado), y el mismo order_id_txt se repite entre países distintos porque cada uno viene de un sistema de origen diferente que no coordina sus IDs con los demás.  

Antes de tocar el código, contesta por escrito:  
6. ¿Qué pasa exactamente cuando dos países generan el mismo order_id_txt con las merge_keys actuales?  
7. ¿Por qué el join contra customers_silver puede estar multiplicando filas, y en qué momento lo arreglarías?  
8. ¿Qué problema concreto trae particionar solo por updated_dt cuando el equipo opera en 9 países?  
9. ¿cuándo SÍ sería seguro usar whenMatchedUpdateAll() tal cual, sin ninguna condición extra?  
Con eso claro, reescribe la función con los tres problemas corregidos.  

### Imports

In [1]:
from src.common.spark_session import get_spark
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark = get_spark("ejercicio2-merge-orders")
spark


:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-99d067f5-f1b0-426e-93ce-0c0fc65bf938;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 204ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

## Respuestas
---

**6. ¿Qué pasa exactamente cuando dos países generan el mismo order_id_txt con las merge_keys actuales?**

En la función se observa que el merge se realiza entre la tabla target "orders_silver" y una tabla enriquecida creada por un join entre las tablas "customer_silver" y "new_batch_df". Estas últimas a su vez presentan duplicados, una por customer_id que provienen de algunos países y la otra presenta order_id_txt duplicados ya que también se repite entre diferentes países. Entonces, cuando dos países generan el mismo order_id_txt con la merge_keys actuales `merge_keys = "order_id_txt"`, podrían ocurrir dos situaciones. La primera, es que si las órdenes llegan por batches separados (en un batch llegan de País_1 y en el otro batch llegan del País_2), puede que un order_id_txt "pise" al que ya existía y lo actualiza, desapareciendo una orden que previamente ya existía.
La segunda situación es que si llegan órdenes de distintos países en un mismo batch, estas generen conflicto y no se pueda realizar el merge.  
Podría considerarse realizar el merge con una merge_keys compuesta, por pais_cd y por order_id_txt. 

---


**7. ¿Por qué el join contra customers_silver puede estar multiplicando filas, y en qué momento lo arreglarías?**

Según el enunciado, "customers_silver" tiene clientes duplicados por customer_id en algunos países debido a un histórico que no se ha limpiado y además, order_id_txt se repite entre países distintos porque cada uno viene de un sistema de origen diferente que no coordina sus IDs.   
Al realizar left join sobre customer_id que contiene a los duplicados en la tabla derecha("customers_silver"), genera una fila de salida por cada duplicado que encuentra. Es decir, si un cliente tiene 2 filas duplicadas en "customers_silver", cada orden proveniente de "new_batch_df", sale multiplicada por esas 2 filas. Además, el join no incluye a pais_cd, por lo que no se solo se multiplicarán por los duplicados de customers_silver, sino también porque customer_id se repite porque proviene de diferentes países.  
Este error podría arreglarse de manera rápida, deduplicando la tabla "customers_silver" antes de realizar left join, teniendo en cuenta customer_id y pais_cd.  
Por otro lado, este error siempre va a estar presente para cualquier tarea que se requiera hacer con "customers_silver", por lo que podría considerarse realizar una limpieza en esta tabla y generar una clave primaria compuesta entre customer_id y pais_cd.

---

**8. ¿Qué problema concreto trae particionar solo por updated_dt cuando el equipo opera en 9 países?**

Actualmente en la función se realiza la partición solo por updated_dt, por lo que los datos de los nueve países quedan mezclados dentro de las mismas particiones por fecha. El problema concreto de particionar solo por updated_dt cuando el equipo opera en 9 países, sería que al momento de hacer una consulta donde se requiera filtrar por país, de todas formas se escaneará la tabla completa, aunque solo se requiera un país en particular. El particionamiento actual no sería de tanta utilidad a nivel rendimiento, lo cual también podría explicar que el equipo de infraestructura se haya quejado de que el job tarda cada vez más.  
Este problema podría arreglarse implementando una partición por pais_cd y por updated_dt. Además, en "new_batch_df" llega update_ts en vez de updated_dt, por lo que habría que transformar esa columna con la función to_date().

---

**9. ¿Cuándo SÍ sería seguro usar whenMatchedUpdateAll() tal cual, sin ninguna condición extra?**

Sería seguro usar whenMatchedUpdateAll() sin ninguna condición extra, cuando se cumpla que la tabla enriquecida y la tabla target tengan exactamente las mismas columnas; cuando la tabla enriquecida no contenga filas duplicadas; cuando tengan un único ID confiable y cuando los datos que vayan a actualizar los existentes sean confiables y actuales. 

---

### Función corregida para el merge incremental de orders_silver

In [ ]:
def merge_orders(spark, new_batch_df, target_table_path): 
    """ Hace merge incremental de nuevas órdenes a la tabla Silver. 
        new_batch_df: order_id_txt, customer_id, pais_cd, monto_dec, updated_ts 
    """ 
   
    target = DeltaTable.forPath(spark, target_table_path)

    # Deduplicamos new_batch_df por order_id_txt y pais_cd, quedandonos con el updated_ts mas reciente
    batch_dedup_df = (
        new_batch_df
        .withColumn("rn", F.row_number().over(
            Window.partitionBy("order_id_txt", "pais_cd").orderBy(F.col("updated_ts").desc())
        ))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )
     
    # Deduplicamos customers_silver por customer_id y pais_cd, quedandonos con el updated_ts mas reciente
    customers_dedup_df = (
        spark.table("customers_silver")
        .withColumn("rn", F.row_number().over(
            Window.partitionBy("customer_id", "pais_cd").orderBy(F.col("updated_ts").desc())
        ))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    # Enriquecemos con datos del cliente
    enriched_df = batch_dedup_df.join( 
        customers_dedup_df, 
        ["customer_id", "pais_cd"], 
        "left" 
    ) 

    # update_ts a updated_dt
    enriched_df = enriched_df.withColumn("updated_dt", F.to_date(F.col("updated_ts")))

    # La orden se identifica por pais + order_id_txt
    merge_keys = ["pais_cd", "order_id_txt"]

    # Si el source contiene la versión completa de la orden, whenMatchedUpdateAll() es apropiado.
    (target.alias("t") 
        .merge( 
            enriched_df.alias("s"), 
            " AND ".join([f"t.{k} = s.{k}" for k in merge_keys])
        ) 
        .whenMatchedUpdateAll() 
        .whenNotMatchedInsertAll()  
        .execute() 
    ) 

   # Particionamiento por país y fecha para evitar mezclar los datos de los distintos países 
    (
        enriched_df.write
            .format("delta") 
            .mode("append") 
            .partitionBy("pais_cd", "updated_dt")
            .save(target_table_path + "_backup") 
    )
    return enriched_df